<div style="padding:22px 26px;border-radius:18px;background:linear-gradient(135deg,#f7f7ff,#eef7ff);border:1px solid #d9e2ff;">
  <h1 style="margin:0;">2026 Quantum Korea Hackathon</h1>
  <h2 style="margin:8px 0 0 0;font-weight:500;">Trotterized Quantum Simulation — 5인 작업계획표</h2>
  <p style="margin:12px 0 0 0;font-size:15px;">
    목표: <b>Hamiltonian 구성 → Trotter 회로 → exact/Trotter error → autocorrelation spectrum → backend/bonus</b>를 병렬로 완성한다.
  </p>
</div>

## 0. 공통 문제 설정

이번 프로젝트의 모델은 periodic anisotropic Heisenberg chain with transverse $X$ field이다.

$$
H=\sum_{j=0}^{N-1}(J_x X_jX_{j+1}+J_yY_jY_{j+1}+J_zZ_jZ_{j+1})+h_x\sum_{j=0}^{N-1}X_j
$$

기본 설정은 다음과 같다.

| 항목 | 값 |
|---|---:|
| qubit number | $N=6$ |
| coupling | $J_x=1.0,\ J_y=0.7,\ J_z=1.2$ |
| transverse field | $h_x=0.4$ |
| initial state | $|\psi_0\rangle=|010101\rangle$ |
| time grid | $t=0,0.05,0.10,\dots,15.0$ |
| time step | $\Delta t=0.05$ |
| max time | $T_{\max}=15.0$ |

<div style="padding:12px 16px;border-left:5px solid #555;background:#f8f8f8;border-radius:8px;">
<b>주의:</b> 모든 팀원이 같은 qubit ordering과 bitstring convention을 써야 한다. 여기서 어긋나면 plot과 spectrum이 전부 달라진다.
</div>

## 1. 전체 역할 요약

| 팀원 | 핵심 담당 | 최종 산출물 |
|---|---|---|
| **조현민** | 팀장, autocorrelation spectroscopy, 결과 통합 | $A(t)$ plot, FFT spectrum, dominant peaks, 최종 스토리 |
| **곽재현** | Qiskit core 구현, Hamiltonian, Trotter circuit, exact baseline | `SparsePauliOp`, Trotter circuit, resource count, exact state |
| **정소연** | Trotter error, resource-error trade-off | state infidelity, $Z_1$ error, $Z_0Z_1$ error, trade-off discussion |
| **신상준** | experiment config, backend, bonus optimization | experiment grid, backend/fake backend results, transpilation table |
| **문채영** | Heisenberg physics, symmetry/physical interpretation | model physics, transverse field 해석, spectral interpretation 보강 |

## 2. 의존성 구조

```text
공통 코드: Hamiltonian / initial state / time grid
    |
    |---- 곽재현: Trotter circuit + resource count
    |          |
    |          |---- 정소연: state/observable error
    |          |
    |          |---- 신상준: backend + bonus transpilation
    |
    |---- 조현민: exact autocorrelation
    |          |
    |          |---- FFT spectrum + dominant peaks
    |
    |---- 문채영: physics/symmetry/spectral interpretation
```

가장 먼저 끝나야 하는 것은 곽재현의 공통 Qiskit core 구현이다. 이후 계산 파트는 대부분 병렬화 가능하다.

# Phase 0. 시작 직후 — 공통 convention 확정

| 순서 | 담당 | 작업 | 산출물 |
|---:|---|---|---|
| 0-1 | 곽재현 | Hamiltonian convention 확정 | qubit ordering, Pauli string convention |
| 0-2 | 곽재현 | $N,J_x,J_y,J_z,h_x$ 설정 | `config.py` 또는 notebook 상수 |
| 0-3 | 곽재현 | $|\psi_0\rangle=|010101\rangle$ 초기상태 생성 | `psi0` |
| 0-4 | 곽재현 | 시간격자 생성 | `t_grid` |
| 0-5 | 조현민 | 전체 notebook/report 목차 생성 | `main.ipynb`, `figures/`, `tables/` |
| 0-6 | 전체 | convention 검수 | 모두 같은 결과 확인 |

# Phase 1. 공통 구현 완료

## 곽재현 — Qiskit core baseline

담당 문제: **Problem 1(a), 1(b), 1(c) 일부**

| 순서 | 작업 | 산출물 |
|---:|---|---|
| 1 | `SparsePauliOp` Hamiltonian 생성 | `make_hamiltonian()` |
| 2 | Pauli term 수 계산 | 총 term count |
| 3 | periodic boundary term 식별 | $(5,0)$ bond의 $XX,YY,ZZ$ |
| 4 | `PauliEvolutionGate` 기반 circuit 생성 함수 작성 | `make_evolution_circuit()` |
| 5 | Lie/Suzuki Trotter circuit 생성 함수 작성 | `make_trotter_circuit(t, method, order, reps)` |
| 6 | exact state simulation 함수 작성 | `exact_state(t)` |

공통 함수 목록:

```python
make_hamiltonian()
make_initial_state()
make_trotter_circuit()
exact_state()
expectation_value()
```

## Problem 1(a)에서 바로 적을 수 있는 내용

Periodic chain의 bond는 다음 6개이다.

$$
(0,1),(1,2),(2,3),(3,4),(4,5),(5,0)
$$

각 bond마다 $XX$, $YY$, $ZZ$ term이 있으므로 interaction term은 $6\times3=18$개이다.  
여기에 field term $X_0,\dots,X_5$가 6개 있으므로 전체 Pauli term 수는 다음과 같다.

$$
18+6=24
$$

Periodic boundary condition에서 오는 term은 마지막 bond $(5,0)$에서 생기는 세 항이다.

$$
X_5X_0,\quad Y_5Y_0,\quad Z_5Z_0
$$

<div style="padding:12px 16px;border-left:5px solid #c47f00;background:#fff8e6;border-radius:8px;">
<b>검수 포인트:</b> Qiskit의 Pauli string 표기 순서가 물리적 qubit index와 반대로 보일 수 있다. 보고서에는 사용한 convention을 명시한다.
</div>

## 신상준 — experiment config 관리

기존 역할인 seed/config 관리를 이번 문제에서는 실험 grid와 파일명 convention 관리로 연결한다.

| 순서 | 작업 | 산출물 |
|---:|---|---|
| 1 | 실험 parameter grid 정리 | `experiment_config.yaml` 또는 dict |
| 2 | Trotter method 목록 정리 | Lie, Suzuki-2, Suzuki-4 |
| 3 | reps 목록 정리 | $k=1,2,3$ |
| 4 | 대표 time point 선정 | $t=0.5,1.0,2.0,5.0$ |
| 5 | 전체 time grid 결과 저장 형식 정의 | CSV schema |
| 6 | plot style/filename convention 관리 | `fig_state_error_*.png` 등 |

권장 experiment grid:

```yaml
methods:
  - LieTrotter: reps = [1, 2, 3]
  - SuzukiTrotter order 2: reps = [1, 2, 3]
  - SuzukiTrotter order 4: reps = [1, 2, 3]

time_grid:
  start: 0.00
  stop: 15.00
  step: 0.05

representative_times:
  - 0.5
  - 1.0
  - 2.0
  - 5.0
```

# Phase 2. 병렬 계산 시작

공통 함수가 공유되면 세 개의 큰 계산 흐름이 동시에 진행된다.

<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;">
  <div style="padding:14px;border:1px solid #ddd;border-radius:12px;background:#fafafa;">
    <b>곽재현</b><br>
    Trotter circuit<br>
    Resource count
  </div>
  <div style="padding:14px;border:1px solid #ddd;border-radius:12px;background:#fafafa;">
    <b>정소연</b><br>
    State error<br>
    Observable error
  </div>
  <div style="padding:14px;border:1px solid #ddd;border-radius:12px;background:#fafafa;">
    <b>조현민</b><br>
    Autocorrelation<br>
    Spectrum
  </div>
</div>

## 곽재현 — Trotter circuit resource count

담당 문제: **Problem 1(b), 1(c)**

| 순서 | 작업 | 산출물 |
|---:|---|---|
| 1 | `LieTrotter(reps=k)` circuit 생성 | $k=1,2,3$ |
| 2 | `SuzukiTrotter(order=2,reps=k)` circuit 생성 | $k=1,2,3$ |
| 3 | `SuzukiTrotter(order=4,reps=k)` circuit 생성 | $k=1,2,3$ |
| 4 | basis gate로 decompose/transpile | `{cx,u3,u1}` 기준 |
| 5 | depth 계산 | resource table |
| 6 | total gate count 계산 | resource table |
| 7 | CX count 계산 | resource table |
| 8 | one-qubit gate count 계산 | resource table |

최종 표 형식:

| method | order | reps | $t$ | depth | total gates | cx count | 1q gates |
|---|---:|---:|---:|---:|---:|---:|---:|
| Lie | 1 | 1 | 0.5 |  |  |  |  |
| Lie | 1 | 2 | 0.5 |  |  |  |  |
| Suzuki | 2 | 1 | 0.5 |  |  |  |  |
| Suzuki | 4 | 1 | 0.5 |  |  |  |  |

## 정소연 — Trotter error + resource-error trade-off

담당 문제: **Problem 1(d), 1(e), 1(f), 1(g)**

| 순서 | 작업 | 산출물 |
|---:|---|---|
| 1 | exact state 받아오기 | $|\psi_{\mathrm{exact}}(t)\rangle$ |
| 2 | PF state 계산 | $|\psi_{\mathrm{PF}}(t)\rangle$ |
| 3 | state infidelity 계산 | $\epsilon_{\mathrm{state}}(t)$ |
| 4 | $Z_1$ expectation error 계산 | $\epsilon_{Z_1}(t)$ |
| 5 | $Z_0Z_1$ expectation error 계산 | $\epsilon_{Z_0Z_1}(t)$ |
| 6 | error plot 생성 | time vs error plots |
| 7 | resource-error trade-off 해석 | Problem 1(g) discussion |

핵심 식:

$$
\epsilon_{\mathrm{state}}(t)
=
1-
|\langle\psi_{\mathrm{exact}}(t)|\psi_{\mathrm{PF}}(t)\rangle|^2
$$

$$
\epsilon_O(t)
=
|\langle O\rangle_{\mathrm{exact}}(t)
-
\langle O\rangle_{\mathrm{PF}}(t)|
$$

여기서 $O=Z_1,\ Z_0Z_1$이다.

## 조현민 — autocorrelation spectroscopy

담당 문제: **Problem 2(a), 2(b), 2(c), 2(d), 2(e)**

| 순서 | 작업 | 산출물 |
|---:|---|---|
| 1 | exact autocorrelation 계산 | $A_{\mathrm{exact}}(t)$ |
| 2 | Re, Im, Abs plot | Problem 2(a) |
| 3 | Trotterized autocorrelation 계산 | $A_{\mathrm{PF}}(t)$ |
| 4 | autocorrelation error 계산 | $\epsilon_A(t)$ |
| 5 | FFT spectrum 계산 | spectrum plot |
| 6 | dominant peaks 추출 | peak table |
| 7 | spectral meaning 설명 작성 | Problem 2(e) |

핵심 식:

$$
A(t)=\langle\psi_0|e^{-iHt}|\psi_0\rangle
$$

Exact eigenbasis에서 전개하면 다음과 같다.

$$
A(t)=\sum_n |\langle E_n|\psi_0\rangle|^2 e^{-iE_nt}
$$

따라서 Fourier transform을 하면 $|\psi_0\rangle$와 overlap이 있는 eigenstate의 eigenenergy 위치에서 peak가 생긴다.

<div style="padding:12px 16px;border-left:5px solid #0066cc;background:#eef6ff;border-radius:8px;">
<b>FFT convention:</b> 신호가 $e^{-iEt}$ 형태이므로 사용하는 FFT convention에 따라 peak가 $E$가 아니라 $-E$ 위치에 나타날 수 있다. 보고서에 convention을 명시한다.
</div>

## 문채영 — Heisenberg physics + symmetry interpretation

담당 문제: **Problem 1(g), Problem 2(e), conclusion 보강**

| 순서 | 작업 | 산출물 |
|---:|---|---|
| 1 | Hamiltonian의 물리적 의미 정리 | model explanation |
| 2 | $XX,YY,ZZ$ interaction과 transverse $X$ field 설명 | physics paragraph |
| 3 | $|010101\rangle$ 초기상태의 의미 설명 | staggered product state |
| 4 | $h_x\sum_jX_j$가 spin flip을 만든다는 점 설명 | symmetry discussion |
| 5 | exact dynamics와 Trotter error 구분 설명 | interpretation |
| 6 | autocorrelation peak의 물리적 의미 보강 | spectrum discussion |

중요한 점: 이번 Hamiltonian에는 transverse $X$ field가 있다.

$$
h_x\sum_j X_j
$$

이 term은 computational basis에서 spin을 flip하므로 일반적인 $Z$-magnetization $M_z$를 보존하지 않는다.  
따라서 단순히 fixed magnetization sector 안에서만 dynamics가 일어난다고 해석하면 안 된다.

## 신상준 — backend/bonus + optional randomized extension

담당 문제: **Problem 1(h), Bonus Problem 3**

| 순서 | 작업 | 산출물 |
|---:|---|---|
| 1 | fake backend 또는 IBM backend 선택 | backend name |
| 2 | 대표 circuit 선택 | e.g. Suzuki order=2, reps=1 |
| 3 | 대표 time points 선택 | $t=0.5,1.0,2.0$ |
| 4 | $\langle Z_1\rangle$, $\langle Z_0Z_1\rangle$ 측정 회로 구성 | measurement circuits |
| 5 | Qiskit Runtime / fake backend 실행 | backend result table |
| 6 | exact simulation과 비교 | discrepancy table |
| 7 | Bonus용 representative circuit 생성 | order=2, reps=3, $t=0.5$ |
| 8 | optimization level 0,1,2,3 transpile | transpilation table |
| 9 | Rustiq 가능 여부 확인 | available/unavailable |
| 10 | 가능하면 Rustiq resource 비교 | bonus comparison |

Problem 1(h) 추천 설정:

$$
\mathrm{SuzukiTrotter(order=2,reps=1)}
$$

Bonus 3 추천 설정:

$$
\mathrm{SuzukiTrotter(order=2,reps=3)},\quad t=0.5
$$

# Phase 3. 시간순 전체 계획

## T0 ~ T+1h: 공통 코드와 구조 확정

| 시간 | 담당 | 작업 |
|---|---|---|
| T0~T+20m | 곽재현 | Hamiltonian, initial state, time grid 구현 |
| T0~T+20m | 조현민 | notebook/report 구조 생성 |
| T+20m~T+40m | 곽재현 | Pauli term count, periodic boundary term 확인 |
| T+20m~T+40m | 신상준 | 실험 grid와 파일명 convention 정리 |
| T+40m~T+1h | 전체 | qubit ordering, bitstring convention 검수 |

## T+1h ~ T+3h: 병렬 구현

| 시간 | 담당 | 작업 |
|---|---|---|
| T+1h~T+3h | 곽재현 | Trotter circuit 생성, resource count 코드 |
| T+1h~T+3h | 정소연 | exact/PF state comparison 코드 |
| T+1h~T+3h | 조현민 | exact autocorrelation 계산 코드 |
| T+1h~T+3h | 신상준 | backend/fake backend 실행 환경 확인 |
| T+1h~T+3h | 문채영 | model physics 설명 초안 작성 |

## T+3h ~ T+5h: 1차 결과 생성

| 시간 | 담당 | 작업 |
|---|---|---|
| T+3h~T+5h | 곽재현 | resource count table 생성 |
| T+3h~T+5h | 정소연 | state infidelity plot 생성 |
| T+3h~T+5h | 조현민 | $A(t)$ Re/Im/Abs plot 생성 |
| T+3h~T+5h | 신상준 | 대표 circuit transpilation 테스트 |
| T+3h~T+5h | 문채영 | 초기상태, transverse field, symmetry 해석 작성 |

## T+5h ~ T+7h: 2차 결과 완성

| 시간 | 담당 | 작업 |
|---|---|---|
| T+5h~T+7h | 정소연 | $Z_1$, $Z_0Z_1$ observable error plot |
| T+5h~T+7h | 조현민 | FFT spectrum, dominant peak table |
| T+5h~T+7h | 신상준 | backend/fake backend result table |
| T+5h~T+7h | 곽재현 | representative circuit diagrams/resource summary 정리 |
| T+5h~T+7h | 문채영 | Problem 2(e) spectral interpretation 보강 |

## T+7h ~ T+8.5h: 해석 파트 작성

| 담당 | 작성할 내용 |
|---|---|
| 곽재현 | Hamiltonian construction, Trotter circuit construction, resource count 설명 |
| 정소연 | Trotter error trend, resource-error trade-off, same reps vs similar depth 비교 |
| 조현민 | autocorrelation spectroscopy, FFT peak 해석, 전체 결과 스토리 |
| 신상준 | backend discrepancy, transpilation optimization, Rustiq 가능 여부 |
| 문채영 | Heisenberg physics, transverse field, 초기상태 의미, symmetry 관련 주의점 |

## T+8.5h ~ T+10h: 통합 및 검수

| 시간 | 담당 | 작업 |
|---|---|---|
| T+8.5h~T+9h | 조현민 | 전체 결과 통합 |
| T+8.5h~T+9h | 곽재현 | 코드 재실행 가능성 확인 |
| T+9h~T+9.5h | 정소연 | error plot과 resource table 수치 일관성 검수 |
| T+9h~T+9.5h | 신상준 | backend/bonus 표 정리 |
| T+9h~T+9.5h | 문채영 | 물리 해석 문장 검수 |
| T+9.5h~T+10h | 전체 | 최종 제출본 확인 |

# Phase 4. 최종 보고서 목차와 담당자 매핑

| 섹션 | 담당 |
|---|---|
| 1. Model and setup | 곽재현 + 문채영 |
| 2. Hamiltonian construction | 곽재현 |
| 3. Trotter circuit construction | 곽재현 |
| 4. Resource count | 곽재현 + 신상준 |
| 5. Exact vs Trotter state error | 정소연 |
| 6. Local observable error | 정소연 |
| 7. Resource-error trade-off | 정소연 |
| 8. Backend experiment | 신상준 |
| 9. Autocorrelation signal | 조현민 |
| 10. Fourier spectrum | 조현민 |
| 11. Spectral/physical interpretation | 조현민 + 문채영 |
| 12. Bonus optimization | 신상준 |
| 13. Conclusion | 조현민 |

# Phase 5. 우선순위

## 무조건 끝내야 하는 것

| 담당 | 필수 작업 |
|---|---|
| 곽재현 | Hamiltonian, Trotter circuit, exact baseline, resource count |
| 정소연 | state infidelity, observable error, trade-off discussion |
| 조현민 | autocorrelation, FFT spectrum, result integration |

## 점수 보강용

| 담당 | 보강 작업 |
|---|---|
| 신상준 | backend/fake backend, transpilation optimization, Rustiq 가능 여부 |
| 문채영 | 물리 해석, transverse field와 symmetry 설명, spectrum interpretation 보강 |

## 시간 남을 때만

| 담당 | optional extension |
|---|---|
| 정소연 | MPF extrapolation |
| 신상준 | randomized product formula / qDRIFT |
| 곽재현 | momentum-sector 또는 HSP-inspired Fourier interpretation |

# Final Checklist

- [ ] 모든 팀원이 같은 $H$, $|\psi_0\rangle$, $t$ grid를 사용했는가?
- [ ] `SparsePauliOp` Pauli string convention을 명시했는가?
- [ ] periodic boundary term $(5,0)$의 $XX,YY,ZZ$를 식별했는가?
- [ ] resource count에서 depth, total gates, CX count, one-qubit count를 모두 보고했는가?
- [ ] exact state와 Trotter state 비교에서 state infidelity를 사용했는가?
- [ ] $Z_1$, $Z_0Z_1$ observable error plot을 넣었는가?
- [ ] autocorrelation의 Re, Im, Abs plot을 넣었는가?
- [ ] FFT convention과 peak sign을 설명했는가?
- [ ] backend 결과와 exact result의 차이를 noise, readout, CX error, transpilation overhead로 설명했는가?
- [ ] bonus optimization level 0,1,2,3 resource table을 넣었는가?